# Pipeline Huấn Luyện & Đánh Giá Validation LightGBM với Loss MAE
Dự án: Tốt nghiệp - Energy Forecasting - Nhóm thực hiện: The Outliers

Notebook này thực hiện trọn vẹn quy trình (**Tune Optuna -> Train Final -> Evaluate Validation 3 Phạm vi**) cho **Loss MAE**.

> ### Lưu ý về bộ nhớ RAM, Hỗ trợ GPU và Log Tiến Trình Optuna
>
> Notebook này thực hiện quy trình **Tune Optuna -> Train Final -> Đánh giá tập VALIDATION** cho 1 hàm loss duy nhất.
>
> **LOG TIẾN TRÌNH RÕ RÀNG:** Hiển thị chi tiết thời gian chạy, số dòng dữ liệu, tiến độ từng fold và từng trial Optuna (kèm thời gian đã chạy & ước tính còn lại ETA).
> **HỖ TRỢ TĂNG TỐC GPU:** Tự động phát hiện driver NVIDIA OpenCL trên NixOS/Linux/Windows. Nếu không có GPU hoặc lỗi driver, tự động fallback sang CPU (`lightgbm_cpu_after_gpu_retry`).
> **NGUYÊN TẮC BAN ĐÊM VÀ NIÊM PHONG TẬP TEST:**
> 1. Tập train có >50% số dòng là ban đêm (sản lượng ~0 kWh). Việc gộp ban đêm vào chấm điểm sẽ làm chỉ số RMSE/R2 bị thổi phồng giả tạo.
> 2. Mô hình vẫn được **huấn luyện trên toàn bộ ngày + đêm** để học thời điểm chuyển giao bình minh/hoàng hôn.
> 3. Nhưng **METRIC CHÍNH THỨC BÁO CÁO** chỉ tính ở phạm vi BAN NGÀY (`is_daylight == True` và `energy_source == "measured"`).
> 4. Notebook này **KHÔNG ĐỤNG TỚI TẬP TEST** (`v3_test_selected.parquet`). Chấm test duy nhất 1 lần tại Notebook 07.

## Bước 2. Import thư viện và khai báo tham số

In [ ]:
LOSS_NAME = 'mae'
LGB_OBJECTIVE = 'regression_l1'

import gc
import json
import os
import pickle
import platform
import statistics
import time
import warnings

import lightgbm as lgb
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
import numpy as np
import optuna
import pandas as pd
import pyarrow.parquet as pq
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Tham số chung ──
VERSION = 'v3'
SITE_COL = 'site_id'
TIMESTAMP_COL = 'timestamp'
TARGET_COL = 'energy_generated_kwh'

FOLDS = [1, 2, 3, 4, 5]
N_TRIALS = 20
SEED = 42
EARLY_STOPPING_ROUNDS = 100

# ── Tham số Cấu hình GPU & Log ──
USE_GPU = True          # Đổi thành False nếu muốn ép chạy CPU
GPU_PLATFORM_ID = 0
GPU_DEVICE_ID = 0
VERBOSE_FOLD = True     # In tiến độ từng fold trong mỗi trial Optuna

# ── Thư mục đầu vào / đầu ra ──
SELECTED_DIR = '../../data/model/v3/05_selected'
BASE_OUTPUT_DIR = '../../data/model/v3/06_train'
OUTPUT_DIR = f'{BASE_OUTPUT_DIR}/{LOSS_NAME}'

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Đã import thư viện và khai báo tham số.")
print(f"- Loss function    : {LOSS_NAME.upper()} (objective: {LGB_OBJECTIVE})")
print(f"- Cấu hình GPU     : USE_GPU={USE_GPU} (platform_id={GPU_PLATFORM_ID}, device_id={GPU_DEVICE_ID})")
print(f"- Log chi tiết fold: VERBOSE_FOLD={VERBOSE_FOLD}")
print(f"- Số trials Optuna : {N_TRIALS}")
print(f"- Early stopping    : {EARLY_STOPPING_ROUNDS}")
print(f"- Đọc dữ liệu từ    : {SELECTED_DIR}")
print(f"- Ghi kết quả ra    : {OUTPUT_DIR}")

## Bước 2.1. Thiết lập GPU (OpenCL) cho LightGBM

In [ ]:
import os
import platform
import subprocess

# OCL_ICD_VENDORS chỉ dành cho Linux. Windows/macOS KHÔNG dùng biến này:
# LightGBM trên Windows tìm OpenCL qua driver của hệ điều hành.
LA_LINUX = (os.name == "posix" and platform.system() == "Linux")

if LA_LINUX:
    OCL_CANDIDATES = [
        "/run/opengl-driver/etc/OpenCL/vendors",   # NixOS: symlink ổn định, tự cập nhật khi đổi driver
        "/etc/OpenCL/vendors",                     # Ubuntu/Debian chuẩn FHS
    ]
    if "OCL_ICD_VENDORS" in os.environ:
        print(f"OCL_ICD_VENDORS đã được đặt sẵn: {os.environ['OCL_ICD_VENDORS']}")
    else:
        for _p in OCL_CANDIDATES:
            if os.path.isdir(_p) and any(f.endswith(".icd") for f in os.listdir(_p)):
                os.environ["OCL_ICD_VENDORS"] = _p
                print(f"Đã tự đặt OCL_ICD_VENDORS = {_p}")
                break
        else:
            print("[CẢNH BÁO] Không tìm thấy thư mục ICD OpenCL trên máy Linux này.")
            print("   LightGBM sẽ chạy CPU. Nếu muốn GPU, cài driver OpenCL rồi chạy lại.")
else:
    print(f"Hệ điều hành: {platform.system()} - bỏ qua OCL_ICD_VENDORS (biến này chỉ cho Linux).")
    print("   LightGBM sẽ tự tìm GPU theo driver của hệ điều hành.")
    print("   Nếu bản LightGBM không được build kèm GPU thì notebook tự động chuyển sang CPU.")

def kiem_tra_gpu():
    """Thử train 1 model nhỏ trên GPU để kiểm tra tính khả dụng."""
    try:
        import lightgbm as lgb
        import numpy as np
        X = np.random.rand(200, 4)
        y = np.random.rand(200)
        lgb.train(
            {"objective": "regression", "device": "gpu", "gpu_platform_id": GPU_PLATFORM_ID, "gpu_device_id": GPU_DEVICE_ID, "verbose": -1},
            lgb.Dataset(X, y),
            num_boost_round=1
        )
        return True, ""
    except Exception as e:
        return False, str(e)[:200]

GPU_SAN_SANG = False
if USE_GPU:
    GPU_SAN_SANG, _err = kiem_tra_gpu()
    if GPU_SAN_SANG:
        print("GPU OpenCL sẵn sàng. LightGBM sẽ chạy trên GPU.")
    else:
        print("[CẢNH BÁO] Không dùng được GPU, tự động chuyển sang CPU.")
        print(f"   Lý do: {_err}")
else:
    print("USE_GPU = False -> Chạy CPU theo cấu hình.")

print(f"Chế độ tính toán chính thức: {'GPU' if GPU_SAN_SANG else 'CPU'}")

### Hướng Dẫn Về Thiết Lập GPU & Khả Năng Tương Thích
- **Máy Linux/NixOS:** notebook tự đặt `OCL_ICD_VENDORS`, không cần export tay.
- **Máy Windows:** bỏ qua biến này; nếu LightGBM không có GPU thì tự động chạy CPU, vẫn cho kết quả bình thường.
- **Cơ chế Fallback An Toàn:** Nếu quá trình huấn luyện GPU gặp sự cố (thiếu VRAM hoặc lỗi OpenCL runtime), mã nguồn sẽ bắt exception và chuyển sang CPU (`lightgbm_cpu_after_gpu_retry`) để đảm bảo pipeline luôn hoàn thành 100%.

## Bước 3. Đọc danh sách đặc trưng đã chọn

In [ ]:
json_path = f'{SELECTED_DIR}/selected_features.json'
with open(json_path, 'r', encoding='utf-8') as f:
    _sel_raw = json.load(f)

selected_features = _sel_raw['selected_features'] if isinstance(_sel_raw, dict) else _sel_raw

# Danh sách cột cần thiết để nạp (tiết kiệm RAM)
NEEDED_COLS = selected_features + [
    TARGET_COL, SITE_COL, TIMESTAMP_COL,
    "energy_source", "exclude_from_training",
    "outlier_group", "has_complete_history_features", "is_daylight"
]


def read_selected(path):
    """Đọc duy nhất các cột cần thiết có trong schema parquet để tiết kiệm RAM."""
    have = set(pq.ParquetFile(path).schema_arrow.names)
    cols = [c for c in NEEDED_COLS if c in have]
    return pd.read_parquet(path, columns=cols)


print(f"Đã đọc {len(selected_features)} đặc trưng từ {json_path}")
print("Đã định nghĩa hàm read_selected tối ưu RAM.")

## Bước 4. Hàm lọc dòng hợp lệ

In [ ]:
def filter_valid_rows(df, name=""):
    """Lọc dòng hợp lệ cho huấn luyện và đánh giá.

    Quy tắc chống rò rỉ và ô nhiễm:
    1. Bo exclude_from_training == True (dữ liệu rò rỉ / gap lớn >= 24h)
    2. Bo has_complete_history_features == False (thiếu lịch sử lag/rolling)
    """
    n_before = len(df)
    out = df.copy()

    if 'exclude_from_training' in out.columns:
        out = out[out['exclude_from_training'] == False]

    if 'has_complete_history_features' in out.columns:
        out = out[out['has_complete_history_features'] == True]

    n_after = len(out)
    n_removed = n_before - n_after
    pct = (n_after / n_before) * 100 if n_before > 0 else 0
    print(f"Lọc dữ liệu {name}: ban đầu {n_before:,} dòng -> còn {n_after:,} dòng (loại {n_removed:,} dòng, {pct:.2f}%)")
    return out


print("Đã định nghĩa hàm filter_valid_rows.")

## Bước 5. PHẦN A: Tune Optuna (Expanding CV gộp 5 Fold)

In [ ]:
# Nạp trước 5 fold vào bộ nhớ 1 lần (cached_folds) để tối ưu tốc độ cho Optuna
cached_folds = []

for fold in FOLDS:
    tr_path = f'{SELECTED_DIR}/time_series_folds/fold_{fold}_train_selected.parquet'
    va_path = f'{SELECTED_DIR}/time_series_folds/fold_{fold}_val_selected.parquet'

    if not os.path.exists(tr_path) or not os.path.exists(va_path):
        print(f"[CẢNH BÁO] Không tìm thấy dữ liệu fold {fold}, bỏ qua.")
        continue

    tr_raw = read_selected(tr_path)
    va_raw = read_selected(va_path)

    tr_df = filter_valid_rows(tr_raw, f"Fold {fold} Train")
    va_df = filter_valid_rows(va_raw, f"Fold {fold} Val")
    del tr_raw, va_raw
    gc.collect()

    feat_cols = [c for c in selected_features if c in tr_df.columns]

    # CHỐT AN TOÀN: Lọc bỏ mọi cột không phải kiểu số
    num_cols = [c for c in feat_cols if pd.api.types.is_numeric_dtype(tr_df[c])]
    drop_non_numeric = [c for c in feat_cols if c not in num_cols]
    if drop_non_numeric:
        print(f"[CẢNH BÁO] Bỏ {len(drop_non_numeric)} cột không phải số khỏi feature: {drop_non_numeric}")
        print("   (kiểm tra lại deny list ở notebook 05 nếu thấy cột phân loại thô ở đây)")
    feat_cols = num_cols

    cat_cols = [c for c in feat_cols if c.endswith('_enc')]

    # TÍNH MEDIAN CHỈ TRÊN TẬP TRAIN CỦA FOLD NÀY (chống rò rỉ)
    medians = tr_df[feat_cols].median(numeric_only=True).fillna(0.0)

    x_tr = tr_df[feat_cols].fillna(medians).astype(np.float32)
    y_tr = tr_df[TARGET_COL].astype(np.float32)
    x_va = va_df[feat_cols].fillna(medians).astype(np.float32)
    y_va = va_df[TARGET_COL].astype(np.float32)

    cached_folds.append({
        'fold': fold,
        'x_train': x_tr,
        'y_train': y_tr,
        'x_val': x_va,
        'y_val': y_va,
        'cat_cols': cat_cols,
    })

    del tr_df, va_df
    gc.collect()

print("")
print(f"Đã nạp thành công {len(cached_folds)} fold vào bộ nhớ cached_folds (dạng float32).")

In [ ]:
def objective(trial):
    """Hàm objective tính POOLED WAPE gộp trên cả 5 fold."""
    obj_choice = LGB_OBJECTIVE

    # Regularization
    reg_type = trial.suggest_categorical("reg_type", ["l1", "l2", "elasticnet"])
    if reg_type == "l1":
        reg_alpha = trial.suggest_float("reg_alpha", 0.0, 10.0)
        reg_lambda = 0.0
    elif reg_type == "l2":
        reg_alpha = 0.0
        reg_lambda = trial.suggest_float("reg_lambda", 0.0, 10.0)
    else:
        reg_alpha = trial.suggest_float("reg_alpha", 0.0, 10.0)
        reg_lambda = trial.suggest_float("reg_lambda", 0.0, 10.0)

    n_est_max = trial.suggest_int("n_estimators", 200, 800)
    learning_rate = trial.suggest_float("learning_rate", 0.01, 0.15, log=True)
    num_leaves = trial.suggest_int("num_leaves", 31, 127)
    min_child_samples = trial.suggest_int("min_child_samples", 20, 200)
    subsample = trial.suggest_float("subsample", 0.7, 1.0)
    colsample_bytree = trial.suggest_float("colsample_bytree", 0.7, 1.0)

    params = {
        'objective': obj_choice,
        'n_estimators': n_est_max,
        'learning_rate': learning_rate,
        'num_leaves': num_leaves,
        'min_child_samples': min_child_samples,
        'subsample': subsample,
        'colsample_bytree': colsample_bytree,
        'reg_alpha': reg_alpha,
        'reg_lambda': reg_lambda,
        'random_state': SEED,
        'n_jobs': -1,
        'verbosity': -1,
    }

    if obj_choice == "huber":
        params['alpha'] = trial.suggest_float("alpha", 0.5, 10.0, log=True)

    # Thêm cấu hình GPU nếu sẵn sàng
    if GPU_SAN_SANG:
        params['device'] = 'gpu'
        params['gpu_platform_id'] = GPU_PLATFORM_ID
        params['gpu_device_id'] = GPU_DEVICE_ID

    abs_err_sum = 0.0
    abs_y_sum = 0.0
    fold_best_iterations = []

    for f_idx, f_payload in enumerate(cached_folds, 1):
        t_fold_start = time.time()
        model = LGBMRegressor(**params)
        cat_cols = f_payload['cat_cols']

        # Fallback thử nghiệm GPU -> CPU
        try:
            model.fit(
                f_payload['x_train'],
                f_payload['y_train'],
                eval_set=[(f_payload['x_val'], f_payload['y_val'])],
                eval_metric='l1',
                callbacks=[
                    early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False),
                    log_evaluation(period=0),
                ],
                categorical_feature=cat_cols if cat_cols else 'auto',
            )
        except Exception as e:
            if params.get('device') == 'gpu':
                print(f"[CẢNH BÁO] Fit trên GPU gặp lỗi (lightgbm_cpu_after_gpu_retry): {str(e)[:100]}. Tự động chuyển sang CPU...")
                cpu_params = params.copy()
                cpu_params['device'] = 'cpu'
                cpu_params.pop('gpu_platform_id', None)
                cpu_params.pop('gpu_device_id', None)
                model = LGBMRegressor(**cpu_params)
                model.fit(
                    f_payload['x_train'],
                    f_payload['y_train'],
                    eval_set=[(f_payload['x_val'], f_payload['y_val'])],
                    eval_metric='l1',
                    callbacks=[
                        early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False),
                        log_evaluation(period=0),
                    ],
                    categorical_feature=cat_cols if cat_cols else 'auto',
                )
            else:
                raise e

        best_iter = int(getattr(model, "best_iteration_", None) or params["n_estimators"])
        fold_best_iterations.append(best_iter)

        pred = model.predict(f_payload['x_val'], num_iteration=best_iter)
        y_val = f_payload['y_val'].to_numpy(dtype=float)

        f_err = float(np.nansum(np.abs(y_val - pred)))
        f_y = float(np.nansum(np.abs(y_val)))

        abs_err_sum += f_err
        abs_y_sum += f_y

        fold_wape = (f_err / f_y * 100.0) if f_y > 0 else float("nan")
        t_fold_elapsed = time.time() - t_fold_start

        if VERBOSE_FOLD:
            print(f"      fold {f_idx}/{len(cached_folds)} | WAPE {fold_wape:.3f}% | best_iter {best_iter} | {t_fold_elapsed:.1f}s")

    if fold_best_iterations:
        trial.set_user_attr("best_iteration_median", int(statistics.median(fold_best_iterations)))

    pooled_wape = (abs_err_sum / abs_y_sum * 100.0) if abs_y_sum > 0 else float("inf")

    # Pruning thủ công Optuna
    trial.report(pooled_wape, step=trial.number)
    if trial.should_prune():
        raise optuna.TrialPruned()

    return pooled_wape


print("Đã định nghĩa hàm objective tính POOLED WAPE gộp (có log tiến độ fold & GPU fallback).")

In [ ]:
# Tính tổng số dòng huấn luyện qua 5 fold
sum_train_rows = sum(len(f['x_train']) for f in cached_folds)
total_train_runs = N_TRIALS * len(cached_folds)

mode_str = 'GPU' if GPU_SAN_SANG else 'CPU'
print(f"--- BẮT ĐẦU TUNE OPTUNA | loss = {LOSS_NAME.upper()} ---")
print(f"Số trial            : {N_TRIALS}")
print(f"Số fold mỗi trial   : {len(cached_folds)}")
print(f"Tổng số lần train   : {total_train_runs}")
print(f"Chế độ tính toán    : {mode_str}")
print(f"Tổng số dòng train  : {sum_train_rows:,}")
print("Ước tính 10-30 phút (tùy máy/GPU). Sẽ in tiến độ chi tiết từng trial...")
print("")

study = optuna.create_study(
    direction="minimize",
    sampler=TPESampler(seed=SEED),
    pruner=MedianPruner(),
)

_t0_study = time.time()


def log_trial(study, trial):
    """Callback in tóm tắt tiến độ sau mỗi trial."""
    elapsed = time.time() - _t0_study
    done = trial.number + 1
    tb_min = elapsed / 60.0
    eta_min = (elapsed / done) * (N_TRIALS - done) / 60.0 if done > 0 else 0.0

    # Lấy giá trị tốt nhất hiện tại
    try:
        best_val = study.best_value
    except ValueError:
        best_val = float("inf")

    # Kiểm tra trial này có phải tốt nhất mới không
    is_new_best = False
    if trial.value is not None and abs(trial.value - best_val) < 1e-12:
        is_new_best = True

    mark_best = " [MỚI TỐT NHẤT]" if is_new_best else ""

    if trial.state == optuna.trial.TrialState.PRUNED or trial.value is None:
        val_str = "(bị prune sớm)"
    else:
        val_str = f"{trial.value:.4f}%"

    p = trial.params
    reg_str = p.get('reg_type', 'none')
    lr_val = p.get('learning_rate', 0.0)
    leaves_val = p.get('num_leaves', 0)

    best_display = f"{best_val:.4f}%" if best_val != float("inf") else "N/A"

    print(
        f"[Trial {done:>2}/{N_TRIALS}] WAPE {val_str:>14} | Tốt nhất {best_display:>8} "
        f"| reg={reg_str} lr={lr_val:.4f} leaves={leaves_val} "
        f"| Đã chạy {tb_min:.1f} phút, còn ~{eta_min:.1f} phút{mark_best}"
    )


study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False, callbacks=[log_trial])

t_study_total = (time.time() - _t0_study) / 60.0

print("")
print(f"--- HOÀN TẤT PHẦN A: TUNE OPTUNA ({t_study_total:.2f} phút) ---")
print(f"Best Pooled WAPE (Cross-Validation) : {study.best_value:.4f}%")
print("Best hyperparameters:")
display(study.best_params)

best_iter_median = study.best_trial.user_attrs.get("best_iteration_median")
final_n_estimators = int(best_iter_median) if best_iter_median else int(study.best_params.get("n_estimators", 500))
print(f"Median best_iteration (final_n_estimators): {final_n_estimators}")

# Ghi trials.csv và best_params.json
trials_df = study.trials_dataframe()
trials_path = f'{OUTPUT_DIR}/optuna_trials.csv'
trials_df.to_csv(trials_path, index=False)

best_params_export = {
    'loss_name': LOSS_NAME,
    'objective': LGB_OBJECTIVE,
    'best_pooled_wape': float(study.best_value),
    'final_n_estimators': final_n_estimators,
    'best_params': study.best_params,
}
json_path = f'{OUTPUT_DIR}/best_params.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(best_params_export, f, ensure_ascii=False, indent=2)

print(f"Đã lưu: {json_path} và {trials_path}")

# Giải phóng cached_folds để dọn RAM cho Phần B
del cached_folds
gc.collect()

## Bước 6. PHẦN B: Huấn luyện Mô hình Cuối cùng trên Development

In [ ]:
print("")
print(f"==================== PHẦN B: TRAIN FINAL MODEL ({LOSS_NAME.upper()}) ====================")

dev_path = f'{SELECTED_DIR}/{VERSION}_development_selected.parquet'
print(f"Đang đọc dữ liệu development từ: {dev_path}")
dev_raw = read_selected(dev_path)

dev_df = filter_valid_rows(dev_raw, "Development")
del dev_raw
gc.collect()

feat_cols = [c for c in selected_features if c in dev_df.columns]

# CHỐT AN TOÀN: Lọc bỏ mọi cột không phải kiểu số
num_cols = [c for c in feat_cols if pd.api.types.is_numeric_dtype(dev_df[c])]
drop_non_numeric = [c for c in feat_cols if c not in num_cols]
if drop_non_numeric:
    print(f"[CẢNH BÁO] Bỏ {len(drop_non_numeric)} cột không phải số khỏi feature: {drop_non_numeric}")
    print("   (kiểm tra lại deny list ở notebook 05 nếu thấy cột phân loại thô ở đây)")
feat_cols = num_cols

cat_cols = [c for c in feat_cols if c.endswith('_enc')]

# TÍNH MEDIAN TRÊN DEVELOPMENT (Lưu vào model_config.json)
feature_medians = dev_df[feat_cols].median(numeric_only=True).fillna(0.0)

X_dev = dev_df[feat_cols].fillna(feature_medians).astype(np.float32)
y_dev = dev_df[TARGET_COL].astype(np.float32)

best_p = study.best_params.copy()
reg_type = best_p.pop('reg_type', None)
best_p.pop('n_estimators', None)
reg_alpha = best_p.pop('reg_alpha', 0.0)
reg_lambda = best_p.pop('reg_lambda', 0.0)

final_params = {
    'objective': LGB_OBJECTIVE,
    'n_estimators': final_n_estimators,
    'reg_alpha': reg_alpha,
    'reg_lambda': reg_lambda,
    'random_state': SEED,
    'n_jobs': -1,
    'verbosity': -1,
    **best_p
}

if GPU_SAN_SANG:
    final_params['device'] = 'gpu'
    final_params['gpu_platform_id'] = GPU_PLATFORM_ID
    final_params['gpu_device_id'] = GPU_DEVICE_ID

t_train_start = time.time()
mode_dev_str = 'GPU' if GPU_SAN_SANG else 'CPU'
print(f"Bắt đầu huấn luyện mô hình cuối với fixed n_estimators={final_n_estimators} trên {len(X_dev):,} dòng (Chế độ: {mode_dev_str})...")
final_model = LGBMRegressor(**final_params)

try:
    final_model.fit(X_dev, y_dev, categorical_feature=cat_cols if cat_cols else 'auto')
except Exception as e:
    if final_params.get('device') == 'gpu':
        print(f"[CẢNH BÁO] Train final model trên GPU gặp lỗi (lightgbm_cpu_after_gpu_retry): {str(e)[:100]}. Tự động chuyển sang CPU...")
        cpu_params = final_params.copy()
        cpu_params['device'] = 'cpu'
        cpu_params.pop('gpu_platform_id', None)
        cpu_params.pop('gpu_device_id', None)
        final_model = LGBMRegressor(**cpu_params)
        final_model.fit(X_dev, y_dev, categorical_feature=cat_cols if cat_cols else 'auto')
    else:
        raise e

t_train_elapsed = time.time() - t_train_start
print(f"Đã huấn luyện xong mô hình cuối trong {t_train_elapsed:.2f} giây ({t_train_elapsed/60.0:.2f} phút)!")

# Ghi model.pkl
model_pkl_path = f'{OUTPUT_DIR}/model.pkl'
with open(model_pkl_path, 'wb') as f:
    pickle.dump(final_model, f)

# Ghi model_config.json
model_config_payload = {
    'loss_name': LOSS_NAME,
    'lgb_objective': LGB_OBJECTIVE,
    'final_n_estimators': final_n_estimators,
    'train_rows': len(dev_df),
    'features': feat_cols,
    'feature_medians': feature_medians.to_dict(),
    'model_params': final_params,
}

config_json_path = f'{OUTPUT_DIR}/model_config.json'
with open(config_json_path, 'w', encoding='utf-8') as f:
    json.dump(model_config_payload, f, ensure_ascii=False, indent=2)

print(f"Đã lưu thành công model.pkl và model_config.json tại: {OUTPUT_DIR}")

del dev_df, X_dev, y_dev
gc.collect()

## Bước 7. PHẦN C: Đánh giá Mô hình trên Tập Validation (3 Phạm Vi)

In [ ]:
print("")
print(f"==================== PHẦN C: EVALUATE VALIDATION 3 PHẠM VI ({LOSS_NAME.upper()}) ====================")

val_path = f'{SELECTED_DIR}/{VERSION}_val_selected.parquet'
print(f"Đang đọc dữ liệu validation từ: {val_path}")
val_raw = read_selected(val_path)

val_df = filter_valid_rows(val_raw, "Validation")
del val_raw
gc.collect()

feat_cols = model_config_payload['features']

# CHỐT AN TOÀN: Lọc bỏ mọi cột không phải kiểu số
num_cols = [c for c in feat_cols if pd.api.types.is_numeric_dtype(val_df[c])]
drop_non_numeric = [c for c in feat_cols if c not in num_cols]
if drop_non_numeric:
    print(f"[CẢNH BÁO] Bỏ {len(drop_non_numeric)} cột không phải số khỏi feature: {drop_non_numeric}")
    print("   (kiểm tra lại deny list ở notebook 05 nếu thấy cột phân loại thô ở đây)")
feat_cols = num_cols

stored_medians = pd.Series(model_config_payload['feature_medians'], dtype=float)

t_eval_start = time.time()
print(f"Bắt đầu dự báo và đánh giá validation trên {len(val_df):,} dòng...")
X_val = val_df[feat_cols].fillna(stored_medians).astype(float)
y_true = val_df[TARGET_COL].values
y_pred = final_model.predict(X_val)

def compute_wape_func(yt, yp):
    abs_y = np.nansum(np.abs(yt))
    return (np.nansum(np.abs(yt - yp)) / abs_y * 100.0) if abs_y > 0 else np.nan

def compute_metrics_func(yt, yp):
    return {
        'wape': compute_wape_func(yt, yp),
        'rmse': root_mean_squared_error(yt, yp),
        'mae': mean_absolute_error(yt, yp),
        'r2': r2_score(yt, yp),
    }

# 1. Phạm vi (a): Tất cả dòng validation (all)
m_all = compute_metrics_func(y_true, y_pred)

# 2. Phạm vi (b): energy_source == "measured"
if 'energy_source' in val_df.columns:
    mask_meas = (val_df['energy_source'] == 'measured').values
else:
    mask_meas = np.ones(len(val_df), dtype=bool)

if mask_meas.sum() > 0:
    m_meas = compute_metrics_func(y_true[mask_meas], y_pred[mask_meas])
else:
    m_meas = {'wape': np.nan, 'rmse': np.nan, 'mae': np.nan, 'r2': np.nan}

# 3. Phạm vi (c): CHÍNH THỨC - energy_source == "measured" AND is_daylight == True
if 'is_daylight' in val_df.columns:
    mask_day = (val_df['is_daylight'] == True).values | (val_df['is_daylight'] == 1).values
    mask_meas_day = mask_meas & mask_day
else:
    print("[CẢNH BÁO] Thiếu cột is_daylight, hãy chạy lại notebook 05! Bỏ qua phạm vi measured_daylight.")
    mask_meas_day = np.zeros(len(val_df), dtype=bool)

if mask_meas_day.sum() > 0:
    m_meas_day = compute_metrics_func(y_true[mask_meas_day], y_pred[mask_meas_day])
else:
    m_meas_day = {'wape': np.nan, 'rmse': np.nan, 'mae': np.nan, 'r2': np.nan}

total_rows = len(val_df)
meas_rows = int(mask_meas.sum())
meas_day_rows = int(mask_meas_day.sum())
t_eval_elapsed = time.time() - t_eval_start

print(f"Đã đánh giá xong validation trong {t_eval_elapsed:.2f} giây!")
print("")
print(f"--- ĐÁNH GIÁ VALIDATION 3 PHẠM VI CHO LOSS {LOSS_NAME.upper()} ---")
print(f"(a) Phạm vi ALL ({total_rows:,} dòng, 100%):")
print(f"    WAPE: {m_all['wape']:.2f}% | RMSE: {m_all['rmse']:.4f} | MAE: {m_all['mae']:.4f} | R2: {m_all['r2']:.4f}")

print("")
print(f"(b) Phạm vi MEASURED ({meas_rows:,} dòng, {meas_rows/total_rows*100:.1f}%):")
print(f"    WAPE: {m_meas['wape']:.2f}% | RMSE: {m_meas['rmse']:.4f} | MAE: {m_meas['mae']:.4f} | R2: {m_meas['r2']:.4f}")

print("")
print(f"(c) Phạm vi MEASURED & DAYLIGHT - METRIC CHÍNH THỨC BÁO CÁO ({meas_day_rows:,} dòng, {meas_day_rows/total_rows*100:.1f}%):")
print(f"    WAPE: {m_meas_day['wape']:.2f}% | RMSE: {m_meas_day['rmse']:.4f} | MAE: {m_meas_day['mae']:.4f} | R2: {m_meas_day['r2']:.4f}")

# Ghi metrics_val.json đầy đủ 3 phạm vi
metrics_val_payload = {
    'loss_name': LOSS_NAME,
    'lgb_objective': LGB_OBJECTIVE,
    'pooled_wape_cv': float(study.best_value),
    'val_total_rows': total_rows,
    'val_measured_rows': meas_rows,
    'val_measured_daylight_rows': meas_day_rows,
    'all': m_all,
    'measured': m_meas,
    'measured_daylight': m_meas_day,
    'scope_a_all': m_all,
    'scope_b_measured': m_meas,
    'scope_c_measured_daylight_official': m_meas_day,
}
metrics_val_path = f'{OUTPUT_DIR}/metrics_val.json'
with open(metrics_val_path, 'w', encoding='utf-8') as f:
    json.dump(metrics_val_payload, f, ensure_ascii=False, indent=2)

print("")
print(f"Đã ghi file đánh giá validation đầy đủ 3 phạm vi: {metrics_val_path}")
del val_df, X_val, y_true, y_pred
gc.collect()

### Lý do loại bỏ ban đêm khỏi Headline Metric & Bảo vệ Niêm phong Tập Test:
1. Hơn 50% số dòng dữ liệu là ban đêm với sản lượng bằng 0. Nếu gộp ban đêm vào chấm điểm, chỉ số RMSE và R2 sẽ bị **làm đẹp giả tạo** vì việc đoán 0 ban đêm là cực kỳ dễ dàng.
2. Mô hình vẫn được huấn luyện trên toàn bộ dữ liệu (ngày + đêm) để học sự chuyển giao giữa bình minh và hoàng hôn.
3. Notebook này **chỉ đánh giá trên tập Validation** để thu thập metrics chọn mô hình.
4. Tập Test vẫn được giữ niêm phong hoàn toàn và **chỉ được chấm duy nhất 1 lần tại Notebook `07_final_test.ipynb`** sau khi đã chọn ra hàm loss thắng trên phạm vi `measured_daylight` của tập Validation.